# Analysis — *Do AI and Human Programmers Fail Differently?*
All paper numbers, tables, and figures come from this notebook, reading `data/classifications/`.
- `full.csv` — gpt-5.4 **judge** (14,182 bugs)
- `gold_human1.csv`,`gold_human2.csv` — two independent **human** annotators (375-item gold)

Verdicts/AC are exact (sandbox); taxonomy distributions use the human gold.

In [1]:
import sys, csv, json, math
from pathlib import Path
from collections import Counter, defaultdict
sys.path.insert(0, str(Path.cwd()))
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src.taxonomy.taxonomy import FAMILIES, TAXONOMY

FAMS=list(FAMILIES); GE=[f for f in FAMS if f.startswith("GE")]; AE=[f for f in FAMS if f.startswith("AE")]
LEAVES=list(TAXONOMY)
FIG=Path("paper/figs"); FIG.mkdir(parents=True,exist_ok=True)
COL={"human":"#4C72B0","clean":"#DD8452","modern":"#55A868"}
ARM={"human":"human","clean":"gpt-3.5-turbo-0125","modern":"gpt-5.4-nano"}
R={}   # results collected for the paper
def labset(s): return set(x.strip() for x in (s or "").split(",") if x.strip() and x.strip()!="UNCOVERED")
def fam(S): return set(x.split(".")[0] for x in S)

full=list(csv.DictReader(open("data/classifications/full.csv"))); fmap={r["item_id"]:r for r in full}
J={r["item_id"]:labset(r["labels"]) for r in full}
h1={r["item_id"]:labset(r["labels"]) for r in csv.DictReader(open("data/classifications/gold_human1.csv"))}
h2={r["item_id"]:labset(r["labels"]) for r in csv.DictReader(open("data/classifications/gold_human2.csv"))}
gids=[i for i in h1 if i in h2 and i in fmap]
R["corpus"]={f"{m}/{s}":n for (m,s),n in sorted(Counter((r["model"],r["stage"]) for r in full).items())}
print("judge bugs:",len(full),"| gold items:",len(gids)); R["corpus"]

judge bugs: 14182 | gold items: 375


{'gpt-3.5-turbo-0125/reflect': 1993,
 'gpt-3.5-turbo-0125/zero_shot': 2076,
 'gpt-5.4-nano/zero_shot': 1465,
 'human/zero_shot': 8648}

## 1. Corpus and acceptance (verdict-based, exact)

In [2]:
def cnt(f): return sum(1 for l in open(f) if l.strip()) if f.exists() else 0
bands=[(0,1199,"<=1199"),(1200,1599,"1200-1599"),(1600,1999,"1600-1999"),(2000,2399,"2000-2399"),(2400,9999,"2400+")]
def band(r):
    r=int(r) if str(r).isdigit() else 0
    for lo,hi,n in bands:
        if lo<=r<=hi: return n
ac={a:[0,0] for a in ARM}; acb={a:defaultdict(lambda:[0,0]) for a in ARM}
rel={"human":"human","clean":"ai/gpt-3.5-turbo-0125","modern":"ai/gpt-5.4-nano"}
for d in sorted(Path("data/problems").glob("*/")):
    if not (d/"meta.json").exists(): continue
    b=band(json.loads((d/"meta.json").read_text()).get("cf_rating") or 0)
    for a in ARM:
        c,i=cnt(d/rel[a]/"correct.jsonl"),cnt(d/rel[a]/"incorrect.jsonl")
        ac[a][0]+=c; ac[a][1]+=i; acb[a][b][0]+=c; acb[a][b][1]+=i
R["ac_rate"]={a:round(100*v[0]/(v[0]+v[1]),1) for a,v in ac.items()}
R["ac_by_band"]={a:{b:round(100*acb[a][b][0]/max(1,sum(acb[a][b])),1) for _,_,b in bands} for a in ARM}
print("AC rate:",R["ac_rate"]); print("AC by band:",json.dumps(R["ac_by_band"],indent=0))
# verdict mix per arm
R["verdict"]={}
for a,m in ARM.items():
    rows=[r for r in full if r["model"]==m and (m!="gpt-3.5-turbo-0125" or r["stage"]=="zero_shot")]
    c=Counter(r["verdict"] for r in rows); t=len(rows)
    R["verdict"][a]={v:round(100*c[v]/t,1) for v in ["WA","TLE","RE","CE"]}
print("verdict mix:",R["verdict"])

AC rate: {'human': 53.4, 'clean': 3.0, 'modern': 31.5}
AC by band: {
"human": {
"<=1199": 59.3,
"1200-1599": 49.5,
"1600-1999": 42.9,
"2000-2399": 55.5,
"2400+": 53.3
},
"clean": {
"<=1199": 7.6,
"1200-1599": 0.8,
"1600-1999": 0.4,
"2000-2399": 3.6,
"2400+": 1.0
},
"modern": {
"<=1199": 55.0,
"1200-1599": 43.8,
"1600-1999": 21.2,
"2000-2399": 35.0,
"2400+": 12.3
}
}
verdict mix: {'human': {'WA': 94.6, 'TLE': 0.9, 'RE': 1.8, 'CE': 2.6}, 'clean': {'WA': 90.0, 'TLE': 1.3, 'RE': 2.6, 'CE': 6.1}, 'modern': {'WA': 93.5, 'TLE': 0.9, 'RE': 0.5, 'CE': 5.1}}


## 2. RQ1 — Human vs AI taxonomy distribution (human gold)

In [3]:
def famcount(src,a,goldonly=True):
    c=Counter()
    for i in (gids if goldonly else src):
        if i not in fmap: continue
        if fmap[i]["model"]!=ARM[a] or (a=="clean" and fmap[i]["stage"]!="zero_shot"): continue
        for x in src[i]: c[x.split(".")[0]]+=1
    return c
def share(src,a):
    c=famcount(src,a); t=sum(c.values()) or 1
    return {f:round(100*c.get(f,0)/t,1) for f in FAMS}
def mean_share(a):  # mean of the two annotators
    return {f:round((share(h1,a)[f]+share(h2,a)[f])/2,1) for f in FAMS}
def chisq(src,a1,a2):
    c1,c2=famcount(src,a1),famcount(src,a2)
    t=np.array([[c1.get(f,0) for f in FAMS],[c2.get(f,0) for f in FAMS]]); t=t[:,t.sum(0)>0]
    chi2,p,dof,_=chi2_contingency(t); n=t.sum()
    return round(chi2,1),float(p),int(dof),round(math.sqrt(chi2/(n*(min(t.shape)-1))),3)
def pooled(a):  # union of both annotators (more power)
    c=Counter()
    for i in gids:
        if fmap[i]["model"]==ARM[a] and (a!="clean" or fmap[i]["stage"]=="zero_shot"):
            for x in (h1[i]|h2[i]): c[x.split(".")[0]]+=1
    return c
def chisq_pooled(a1,a2):
    c1,c2=pooled(a1),pooled(a2); t=np.array([[c1.get(f,0) for f in FAMS],[c2.get(f,0) for f in FAMS]]); t=t[:,t.sum(0)>0]
    chi2,p,dof,_=chi2_contingency(t); n=t.sum(); return round(chi2,1),float(p),round(math.sqrt(chi2/(n*(min(t.shape)-1))),3)

R["family_gold"]={a:mean_share(a) for a in ARM}
R["rq1_stats"]={}
for ann,src in [("a1",h1),("a2",h2)]:
    for a in ("clean","modern"):
        R["rq1_stats"][f"{ann}_human_vs_{a}"]=chisq(src,"human",a)
for a in ("clean","modern"):
    ch,p,v=chisq_pooled("human",a); R["rq1_stats"][f"pooled_human_vs_{a}"]=(ch,p,v)
print("family share (mean of 2 annotators):")
for f in FAMS: print(f"  {f} {FAMILIES[f][:24]:24s} human={R['family_gold']['human'][f]:5} clean={R['family_gold']['clean'][f]:5} modern={R['family_gold']['modern'][f]:5}")
print("\nchi-square:")
for k,v in R["rq1_stats"].items(): print(f"  {k}: chi2={v[0]} p={v[1]:.3g} V={v[-1]} {'SIG' if v[1]<0.05 else 'n.s.'}")
R["ge_share"]={a:round(sum(R["family_gold"][a][f] for f in GE)) for a in ARM}
print("\nGE share (gold):",R["ge_share"])

family share (mean of 2 annotators):
  GE1 Design-related Errors    human= 20.0 clean= 28.0 modern= 31.9
  GE2 Boundary-related Errors  human= 10.8 clean=  6.8 modern=  4.8
  GE3 Condition-related Errors human=  3.6 clean=  1.6 modern=  2.4
  GE4 Data Type Errors         human=  2.0 clean=  0.8 modern=  0.4
  GE5 Syntax Errors            human=  9.2 clean=  8.4 modern=  9.2
  GE6 Input/Output Errors      human=  6.4 clean=  0.8 modern=  0.8
  AE1 Mathematical Problem-rel human= 32.4 clean= 35.2 modern= 33.4
  AE2 Greedy Algorithm-related human=  7.6 clean= 12.8 modern=  9.3
  AE3 Graph Theory-related Err human=  0.0 clean=  0.0 modern=  0.0
  AE4 Recursion & Divide-and-C human=  0.0 clean=  0.0 modern=  0.0
  AE5 Dynamic Programming-rela human=  8.0 clean=  5.6 modern=  7.7
  AE6 Search-related Errors    human=  0.0 clean=  0.0 modern=  0.0

chi-square:
  a1_human_vs_clean: chi2=14.3 p=0.0754 V=0.239 n.s.
  a1_human_vs_modern: chi2=13.5 p=0.096 V=0.231 n.s.
  a2_human_vs_clean: chi2=11

In [4]:
# Fig: family distribution by author (mean of 2 annotators)
fig,ax=plt.subplots(figsize=(11,4)); x=np.arange(len(FAMS)); w=0.27
for i,a in enumerate(ARM):
    ax.bar(x+(i-1)*w,[R["family_gold"][a][f] for f in FAMS],w,label=a,color=COL[a])
ax.set_xticks(x); ax.set_xticklabels(FAMS,fontsize=8); ax.axvline(5.5,color="grey",ls=":",lw=1)
ax.text(2.5,ax.get_ylim()[1]*0.9,"General (implementation)",ha="center",fontsize=9,color="grey")
ax.text(8.5,ax.get_ylim()[1]*0.9,"Algorithm-specific",ha="center",fontsize=9,color="grey")
ax.set_ylabel("% of bug labels (human gold)"); ax.legend(frameon=False)
ax.set_title("Bug-family distribution by author (human gold)"); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG/"family_dist_gold.pdf"); plt.close(fig)
# Fig: GE vs AE share
fig,ax=plt.subplots(figsize=(5,3.6)); aa=list(ARM)
ge=[R["ge_share"][a] for a in aa]; ae=[100-g for g in ge]
ax.bar(aa,ge,label="General (GE)",color="#4C72B0"); ax.bar(aa,ae,bottom=ge,label="Algorithm-specific (AE)",color="#C44E52")
for i,a in enumerate(aa):
    ax.text(i,ge[i]/2,f"{ge[i]}",ha="center",color="white"); ax.text(i,ge[i]+ae[i]/2,f"{ae[i]}",ha="center",color="white")
ax.set_ylabel("% of bug labels"); ax.legend(frameon=False,loc="lower center",bbox_to_anchor=(0.5,-0.33),ncol=2)
ax.set_title("General vs algorithm-specific"); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG/"ge_ae_share.pdf"); plt.close(fig)
print("saved family_dist_gold.pdf, ge_ae_share.pdf")

saved family_dist_gold.pdf, ge_ae_share.pdf


## 3. RQ2 — Difficulty (AC curve) and failure-mode mix

In [5]:
fig,ax=plt.subplots(figsize=(6,4)); bn=[b for _,_,b in bands]
for a in ARM: ax.plot(bn,[R["ac_by_band"][a][b] for b in bn],"o-",label=a,color=COL[a])
ax.set_ylabel("per-submission AC rate (%)"); ax.set_xlabel("difficulty (cf_rating)")
ax.legend(frameon=False); ax.tick_params(axis="x",labelrotation=20); ax.set_title("Acceptance vs difficulty")
ax.spines[["top","right"]].set_visible(False); fig.tight_layout(); fig.savefig(FIG/"ac_difficulty.pdf"); plt.close(fig)
print("saved ac_difficulty.pdf | verdict mix:",R["verdict"])

saved ac_difficulty.pdf | verdict mix: {'human': {'WA': 94.6, 'TLE': 0.9, 'RE': 1.8, 'CE': 2.6}, 'clean': {'WA': 90.0, 'TLE': 1.3, 'RE': 2.6, 'CE': 6.1}, 'modern': {'WA': 93.5, 'TLE': 0.9, 'RE': 0.5, 'CE': 5.1}}


## 4. RQ3 — Self-reflection (verdict-based + distribution shift)

In [6]:
import hashlib
fix=defaultdict(lambda:[0,0])
for d in sorted(Path("data/problems").glob("*/")):
    rf=d/"ai/gpt-3.5-turbo-0125/reflect.jsonl"
    if not rf.exists(): continue
    for line in open(rf):
        r=json.loads(line); iid=hashlib.sha1(f"gpt-3.5-turbo-0125:{d.name}:{r['idx']}".encode()).hexdigest()[:10]
        v=fmap.get(iid,{}).get("verdict","WA"); fix[v][1]+=1; fix[v][0]+=int(r["fixed"])
R["reflect_fix"]={"overall":[sum(x[0] for x in fix.values()),sum(x[1] for x in fix.values())],
                  "by_verdict":{v:fix[v] for v in ["CE","WA","RE","TLE"]}}
print("fix:",R["reflect_fix"]["overall"], R["reflect_fix"]["by_verdict"])
fig,ax=plt.subplots(figsize=(5,3.6)); order=["CE","WA","TLE","RE"]; vals=[100*fix[v][0]/max(1,fix[v][1]) for v in order]
ax.bar(order,vals,color="#8172B3")
for i,v in enumerate(order): ax.text(i,vals[i]+0.2,f"{fix[v][0]}/{fix[v][1]}",ha="center",fontsize=8)
ax.set_ylabel("% bugs fixed"); ax.set_title("Self-reflection fix rate by verdict (clean)"); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG/"reflection_fix.pdf"); plt.close(fig)
# distribution shift (judge labels, gpt-3.5 zero vs reflect)
def jdist(stage):
    c=Counter()
    for r in full:
        if r["model"]=="gpt-3.5-turbo-0125" and r["stage"]==stage:
            for x in labset(r["labels"]): c[x.split(".")[0]]+=1
    t=sum(c.values()); return {f:round(100*c.get(f,0)/t,1) for f in FAMS}
pre,post=jdist("zero_shot"),jdist("reflect")
R["reflect_shift_ge"]=[round(sum(pre[f] for f in GE)),round(sum(post[f] for f in GE))]
fig,ax=plt.subplots(figsize=(11,3.6)); x=np.arange(len(FAMS)); w=0.4
ax.bar(x-w/2,[pre[f] for f in FAMS],w,label="zero-shot (2,076)",color="#DD8452")
ax.bar(x+w/2,[post[f] for f in FAMS],w,label="after reflection (1,993)",color="#8172B3")
ax.set_xticks(x); ax.set_xticklabels(FAMS,fontsize=8); ax.set_ylabel("% of bug labels (judge)")
ax.legend(frameon=False); ax.set_title("Clean-model bug families: before vs after self-reflection")
ax.spines[["top","right"]].set_visible(False); fig.tight_layout(); fig.savefig(FIG/"reflection_shift.pdf"); plt.close(fig)
print("saved reflection_fix.pdf, reflection_shift.pdf | GE share zero->reflect:",R["reflect_shift_ge"])

fix: [83, 2076] {'CE': [7, 127], 'WA': [76, 1868], 'RE': [0, 54], 'TLE': [0, 27]}
saved reflection_fix.pdf, reflection_shift.pdf | GE share zero->reflect: [61, 61]


## 5. RQ4 — Reliability (Cohen's kappa) and Wei comparison

In [7]:
def kappa(A,B,cats,uf):
    n=len(gids)*len(cats); a=b=cc=d=0
    for i in gids:
        sa=fam(A[i]) if uf else A[i]; sb=fam(B[i]) if uf else B[i]
        for x in cats:
            xa,xb=x in sa,x in sb
            a+=xa and xb; b+=xa and not xb; cc+=(not xa)and xb; d+=(not xa)and(not xb)
    po=(a+d)/n; pe=((a+b)*(a+cc)+(cc+d)*(b+d))/(n*n); return round((po-pe)/(1-pe),3)
def f1(A,B):
    tp=fp=fn=0
    for i in gids: tp+=len(A[i]&B[i]); fp+=len(B[i]-A[i]); fn+=len(A[i]-B[i])
    P=tp/max(1,tp+fp); Rc=tp/max(1,tp+fn); return round(2*P*Rc/max(1e-9,P+Rc),3)
R["kappa"]={}
for nm,A,B in [("human1_human2",h1,h2),("judge_human1",J,h1),("judge_human2",J,h2)]:
    R["kappa"][nm]={"leaf":kappa(A,B,LEAVES,0),"family":kappa(A,B,FAMS,1),"microF1":f1(A,B)}
print("kappa / F1:"); [print(f"  {k}: {v}") for k,v in R["kappa"].items()]
R["wei"]={"GE":63.3,"AE":36.7,"GE1":28.6,"GE2":15.5,"GE3":8.7,"GE4":4.3,"GE5":3.5,"GE6":1.9,"AE1":8.1}
print("\nWei et al. distribution:",R["wei"])
Path("paper/results.json").write_text(json.dumps(R,indent=1)); print("\nwrote paper/results.json")

kappa / F1:
  human1_human2: {'leaf': 0.796, 'family': 0.828, 'microF1': 0.802}
  judge_human1: {'leaf': 0.754, 'family': 0.768, 'microF1': 0.762}
  judge_human2: {'leaf': 0.766, 'family': 0.807, 'microF1': 0.773}

Wei et al. distribution: {'GE': 63.3, 'AE': 36.7, 'GE1': 28.6, 'GE2': 15.5, 'GE3': 8.7, 'GE4': 4.3, 'GE5': 3.5, 'GE6': 1.9, 'AE1': 8.1}



wrote paper/results.json


## 6. Leaf-level top labels and dataset characteristics


In [8]:
from collections import Counter
def topleaves(m):
    c=Counter()
    for i in gids:
        if fmap[i]['model']==m and (m!='gpt-3.5-turbo-0125' or fmap[i]['stage']=='zero_shot'):
            for src in (h1,h2):
                for x in src[i]: c[x]+=1
    t=sum(c.values()); return [(l,round(100*n/t)) for l,n in c.most_common(4)]
R['top_leaves']={a:topleaves(m) for a,m in ARM.items()}
print('top leaves per arm (gold):')
for a in ARM: print(' ',a,R['top_leaves'][a])
# difficulty distribution of the 107 problems
dc=Counter()
for d in Path('data/problems').glob('*/meta.json'):
    r=json.loads(d.read_text()).get('cf_rating') or 0
    dc[band(r)]+=1
R['difficulty_dist']={b:dc[b] for _,_,b in bands}
print('difficulty distribution:',R['difficulty_dist'])
Path('paper/results.json').write_text(json.dumps(R,indent=1)); print('updated results.json')


top leaves per arm (gold):
  human [('AE1.1', 30), ('GE5.1', 9), ('GE1.3', 8), ('AE5.2', 8)]
  clean [('AE1.1', 34), ('GE1.2', 14), ('AE2.1', 13), ('GE5.1', 8)]
  modern [('AE1.1', 33), ('GE1.2', 13), ('GE1.1', 10), ('GE5.1', 9)]
difficulty distribution: {'<=1199': 25, '1200-1599': 12, '1600-1999': 13, '2000-2399': 22, '2400+': 35}
updated results.json


## 7. RQ3 — verdict transitions across self-reflection rounds (alluvial)


In [9]:
import yaml
from collections import Counter
from matplotlib.path import Path as MPath
import matplotlib.patches as mpatches
from src.execute.validator import Validator
V=Validator(yaml.safe_load(open("config.yaml")))
VORD=["AC","WA","TLE","RE","CE"]
VCOL={"AC":"#5BA776","WA":"#E2A33C","TLE":"#8172B3","RE":"#CB5B57","CE":"#9AA0A6"}
stages=["Initial","Round 1","Round 2","Round 3"]
def band3(r):
    r=r or 0
    return "Easy" if r<=1599 else "Medium" if r<=2399 else "Hard"
def collect(fb=None):
    seqs=[]
    for d in sorted(Path("data/problems").glob("*")):
        if not (d/"meta.json").exists(): continue
        rating=json.loads((d/"meta.json").read_text()).get("cf_rating") or 0
        if fb and band3(rating)!=fb: continue
        sub=d/"ai/gpt-3.5-turbo-0125"; inc=sub/"incorrect.jsonl"; rf=sub/"reflect.jsonl"
        if not (inc.exists() and rf.exists()): continue
        srcs=[json.loads(l)["source"] for l in inc.read_text().splitlines() if l]
        for line in rf.read_text().splitlines():
            if not line: continue
            r=json.loads(line); init=V.judge(str(d),srcs[r["idx"]])["verdict"]
            tr=[t["verdict"] for t in r["trajectory"]]
            while len(tr)<3: tr.append(tr[-1])
            seqs.append([init]+tr[:3])
    return seqs
def alluvial(ax,seqs,fs=6.0):
    col=[Counter(s[i] for s in seqs) for i in range(4)]
    flows=[Counter((s[i],s[i+1]) for s in seqs) for i in range(3)]
    GAP=0.02; NW=0.05
    def layout(c):
        tot=sum(c.values()); pres=[v for v in VORD if c[v]>0]
        H=1-GAP*(len(pres)-1); y=1.0; lay={}
        for v in pres:
            h=H*c[v]/tot; lay[v]=(y-h,y); y-=h+GAP
        return lay
    lays=[layout(c) for c in col]; xs=[0,1,2,3]
    def ribbon(x0,x1,y0t,y0b,y1t,y1b,color):
        cx=(x0+x1)/2
        verts=[(x0,y0t),(cx,y0t),(cx,y1t),(x1,y1t),(x1,y1b),(cx,y1b),(cx,y0b),(x0,y0b),(x0,y0t)]
        codes=[MPath.MOVETO,MPath.CURVE4,MPath.CURVE4,MPath.CURVE4,MPath.LINETO,MPath.CURVE4,MPath.CURVE4,MPath.CURVE4,MPath.CLOSEPOLY]
        ax.add_patch(mpatches.PathPatch(MPath(verts,codes),fc=color,alpha=0.42,ec="none",zorder=1))
    for i in range(3):
        x0=xs[i]+NW; x1=xs[i+1]
        out={v:lays[i][v][1] for v in lays[i]}; inn={v:lays[i+1][v][1] for v in lays[i+1]}
        for a in VORD:
            if col[i][a]==0: continue
            ah=lays[i][a][1]-lays[i][a][0]
            for b in VORD:
                n=flows[i][(a,b)]
                if n==0: continue
                wa=ah*n/col[i][a]; y0t=out[a]; y0b=out[a]-wa; out[a]-=wa
                bh=lays[i+1][b][1]-lays[i+1][b][0]; wb=bh*n/col[i+1][b]
                y1t=inn[b]; y1b=inn[b]-wb; inn[b]-=wb
                ribbon(x0,x1,y0t,y0b,y1t,y1b,VCOL[a])
    for x,lay,c in zip(xs,lays,col):
        for v,(yb,yt) in lay.items():
            ax.add_patch(mpatches.Rectangle((x,yb),NW,yt-yb,fc=VCOL[v],ec="white",lw=.5,zorder=3))
            if (yt-yb)>0.028:
                ax.text(x+NW/2,(yb+yt)/2,f"{v} {c[v]}",ha="center",va="center",fontsize=fs,zorder=4,
                        bbox=dict(boxstyle="round,pad=0.1",fc="white",ec="none",alpha=0.75))
    for x,st in zip(xs,stages):
        ax.text(x+NW/2,1.04,st,ha="center",fontsize=fs+1,fontweight="bold")
    ax.set_xlim(-0.05,3.2); ax.set_ylim(-0.03,1.10); ax.axis("off")
    return col
# overall
allseq=collect()
rf=defaultdict(lambda:[0,0])
for s in allseq: rf[s[0]][1]+=1; rf[s[0]][0]+=int(s[3]=='AC')
R['reflect_fix']={'overall':[sum(x[0] for x in rf.values()),len(allseq)],'by_verdict':{v:rf[v] for v in ['CE','WA','RE','TLE']}}
print('reflect_fix(consistent):',R['reflect_fix']['by_verdict'])
fig,ax=plt.subplots(figsize=(7.4,4.3)); col=alluvial(ax,allseq,fs=6.5)
plt.tight_layout(); plt.savefig("paper/figs/reflection_sankey.pdf",bbox_inches="tight"); plt.show()
R["reflect_sankey"]={"stages":stages,"N":len(allseq),"col":[{v:c[v] for v in VORD} for c in col]}
print("overall AC/stage:",[col[i]["AC"] for i in range(4)],"N",len(allseq))
# per difficulty (3 panels)
fig,axes=plt.subplots(1,3,figsize=(11,3.7)); R["reflect_sankey_diff"]={}
for ax,b in zip(axes,["Easy","Medium","Hard"]):
    sq=collect(b); c=alluvial(ax,sq,fs=5.3); n=len(sq); ac=c[3]["AC"]
    lab={"Easy":"Easy ($\\leq$1599)","Medium":"Medium (1600--2399)","Hard":"Hard (2400+)"}[b]
    ax.set_title(f"{lab}\nn={n}, fixed {ac} ({100*ac/n:.1f}%)",fontsize=9)
    R["reflect_sankey_diff"][b]={"n":n,"ac":[c[i]["AC"] for i in range(4)],
        "tle":[c[i]["TLE"] for i in range(4)],"re":[c[i]["RE"] for i in range(4)],"ce":[c[i]["CE"] for i in range(4)]}
plt.tight_layout(); plt.savefig("paper/figs/reflection_sankey_diff.pdf",bbox_inches="tight"); plt.show()
Path("paper/results.json").write_text(json.dumps(R,indent=1)); print("results.json updated")
print("diff AC/stage:",{b:R["reflect_sankey_diff"][b]["ac"] for b in R["reflect_sankey_diff"]})


reflect_fix(consistent): {'CE': [7, 127], 'WA': [76, 1867], 'RE': [0, 55], 'TLE': [0, 27]}
overall AC/stage: [0, 53, 73, 83] N 2076


/var/folders/p_/kvwftlzs17b5_3k6yqs8tb300000gn/T/ipykernel_49796/467223541.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("paper/figs/reflection_sankey.pdf",bbox_inches="tight"); plt.show()


results.json updated
diff AC/stage: {'Easy': [0, 30, 43, 50], 'Medium': [0, 15, 20, 23], 'Hard': [0, 8, 10, 10]}


/var/folders/p_/kvwftlzs17b5_3k6yqs8tb300000gn/T/ipykernel_49796/467223541.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("paper/figs/reflection_sankey_diff.pdf",bbox_inches="tight"); plt.show()


## 8. RQ4 — judge vs human family confusion (disagreement structure)


In [10]:
def fam(l): return l.split(".")[0]
fams=[f"GE{i}" for i in range(1,7)]+[f"AE{i}" for i in range(1,7)]
both=Counter(); jonly=Counter(); honly=Counter()
for i in gids:
    jf={fam(x) for x in labset(fmap[i]["labels"])}
    for h in (h1,h2):
        hf={fam(x) for x in h[i]}
        for f in fams:
            if f in jf and f in hf: both[f]+=1
            elif f in jf: jonly[f]+=1
            elif f in hf: honly[f]+=1
R["judge_confusion"]={f:{"both":both[f],"judge_only":jonly[f],"human_only":honly[f]} for f in fams if both[f]+jonly[f]+honly[f]>0}
print("judge over-applies (judge_only>human_only):",
      sorted([(f,jonly[f]-honly[f]) for f in fams], key=lambda x:-x[1])[:3])
print("human over-applies:",
      sorted([(f,honly[f]-jonly[f]) for f in fams], key=lambda x:-x[1])[:3])
Path("paper/results.json").write_text(json.dumps(R,indent=1)); print("saved")


judge over-applies (judge_only>human_only): [('GE1', 53), ('GE2', 24), ('AE5', 9)]
human over-applies: [('AE1', 54), ('GE3', 17), ('GE4', 2)]
saved


## 9. RQ1 robustness: full-corpus distribution, clustered bootstrap, labeler dependence


In [11]:
# === Deconfounding the apparent GE1 "sign flip": labeler vs sample (reviewer must-fix a) ===
import csv as _csv, random as _rnd
from collections import defaultdict as _dd
_rnd.seed(0)
_fam=lambda s:s.split(".")[0]
_full={r["item_id"]:r for r in _csv.DictReader(open("data/classifications/full.csv"))}
_h1={r["item_id"]:r for r in _csv.DictReader(open("data/classifications/gold_human1.csv"))}
_h2={r["item_id"]:r for r in _csv.DictReader(open("data/classifications/gold_human2.csv"))}
_LS=lambda s:set(x for x in (s or "").split(",") if x and x!="UNCOVERED")
_arm={"human":"human","gpt-3.5-turbo-0125":"clean","gpt-5.4-nano":"modern"}
_g=[i for i in _h1 if i in _h2 and i in _full]
_garm=_dd(list)
for i in _g:
    a=_arm.get(_full[i]["model"]);  _garm[a].append(i) if a else None
def _sh(items,getl,F): return 100*sum(1 for i in items if F in {_fam(x) for x in getl(i)})/len(items)
LAB={}  # GE1/GE2 share per arm: human-annot(mean) , judge-on-gold , judge-only%
for F in ["GE1","GE2"]:
    LAB[F]={}
    for a in ["human","clean","modern"]:
        it=_garm[a]
        hm=(_sh(it,lambda i:_LS(_h1[i]['labels']),F)+_sh(it,lambda i:_LS(_h2[i]['labels']),F))/2
        jj=_sh(it,lambda i:_LS(_full[i]['labels']),F)
        jonly=100*sum(1 for i in it if F in {_fam(x) for x in _LS(_full[i]['labels'])} and F not in {_fam(x) for x in (_LS(_h1[i]['labels'])|_LS(_h2[i]['labels']))})/len(it)
        LAB[F][a]={"human":round(hm),"judge_gold":round(jj),"judge_only":round(jonly)}
# judge on FULL corpus per arm (compute inline from full.csv)
_ARMf={"human":("human","zero_shot"),"clean":("gpt-3.5-turbo-0125","zero_shot"),"modern":("gpt-5.4-nano","zero_shot")}
_nf=_dd(int); _g1=_dd(int)
for r in _full.values():
    a=next((k for k,(m,st) in _ARMf.items() if r["model"]==m and r["stage"]==st),None)
    if not a: continue
    _nf[a]+=1; _g1[a]+= int("GE1" in {_fam(x) for x in _LS(r["labels"])})
judge_full={a:round(100*_g1[a]/_nf[a]) for a in ["human","clean","modern"]}
R["deconf"]={"same_items":LAB,"judge_full_GE1":judge_full}
print("GE1 same-items human-annot:",{a:LAB["GE1"][a]["human"] for a in LAB["GE1"]})
print("GE1 same-items judge      :",{a:LAB["GE1"][a]["judge_gold"] for a in LAB["GE1"]})
print("GE1 judge full-corpus     :",judge_full)
print("GE1 judge-only per arm    :",{a:LAB["GE1"][a]["judge_only"] for a in LAB["GE1"]})
# figure: (a) labeler effect (same items), (b) sample effect (judge)
fig,ax=plt.subplots(1,2,figsize=(8.4,3.4)); arms=["human","clean","modern"]; labs=["Human","Clean","Modern"]; x=range(3); w=0.38
hum=[LAB["GE1"][a]["human"] for a in arms]; jg=[LAB["GE1"][a]["judge_gold"] for a in arms]; jf=[judge_full[a] for a in arms]
b1=ax[0].bar([i-w/2 for i in x],hum,w,label="Human annotators",color="#4C72B0")
b2=ax[0].bar([i+w/2 for i in x],jg,w,label="LLM judge",color="#C44E52")
ax[0].set_title("(a) Labeler effect (same 375 gold items)"); ax[0].set_ylabel("GE1 design share (%)")
ax[0].set_xticks(list(x)); ax[0].set_xticklabels(labs); ax[0].legend(fontsize=7); ax[0].set_ylim(0,60)
b3=ax[1].bar([i-w/2 for i in x],jg,w,label="Judge on gold (n=375)",color="#C44E52")
b4=ax[1].bar([i+w/2 for i in x],jf,w,label="Judge on full corpus",color="#8172B3")
ax[1].set_title("(b) Sample effect (LLM judge)"); ax[1].set_xticks(list(x)); ax[1].set_xticklabels(labs)
ax[1].legend(fontsize=7); ax[1].set_ylim(0,60)
for bb in [b1,b2,b3,b4]:
    for r in bb: ax[0 if bb in (b1,b2) else 1].text(r.get_x()+r.get_width()/2,r.get_height()+0.6,f"{r.get_height():.0f}",ha="center",fontsize=7)
plt.tight_layout(); plt.savefig("paper/figs/labeler_dependence.pdf",bbox_inches="tight")
plt.savefig("/private/tmp/claude-501/-Users-cdw-VSCode-aitaxo/46150c7a-bb41-41e4-8f63-7af94d2a5966/scratchpad/deconf.png",dpi=130,bbox_inches="tight"); plt.show()
Path("paper/results.json").write_text(json.dumps(R,indent=1)); print("saved deconf")


GE1 same-items human-annot: {'human': 20, 'clean': 28, 'modern': 32}
GE1 same-items judge      : {'human': 26, 'clean': 34, 'modern': 41}
GE1 judge full-corpus     : {'human': 51, 'clean': 45, 'modern': 52}
GE1 judge-only per arm    : {'human': 6, 'clean': 8, 'modern': 10}


saved deconf


/var/folders/p_/kvwftlzs17b5_3k6yqs8tb300000gn/T/ipykernel_49796/3259925787.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig("/private/tmp/claude-501/-Users-cdw-VSCode-aitaxo/46150c7a-bb41-41e4-8f63-7af94d2a5966/scratchpad/deconf.png",dpi=130,bbox_inches="tight"); plt.show()


## 10. RQ1 power / MDE and exact full-corpus significance (reviewer response)


In [12]:
from scipy.stats import chi2 as _chi2, ncx2 as _ncx2
import math as _m
def _chipV(a,b,pres,fams):
    ta=sum(pres[a][f] for f in fams); tb=sum(pres[b][f] for f in fams); chi=0; k=0
    for f in fams:
        oa,ob=pres[a][f],pres[b][f]
        if oa+ob==0: continue
        k+=1; ea=(oa+ob)*ta/(ta+tb); eb=(oa+ob)*tb/(ta+tb)
        chi+=(oa-ea)**2/ea+(ob-eb)**2/eb
    df=k-1; N=ta+tb
    return chi,df,N,_m.sqrt(chi/N),float(_chi2.sf(chi,df))
def _mde(df,N,power=0.8,alpha=0.05):
    crit=_chi2.ppf(1-alpha,df); lo,hi=1e-4,1.0
    for _ in range(60):
        w=(lo+hi)/2
        (lo,hi)=(w,hi) if float(_ncx2.sf(crit,df,w*w*N))<power else (lo,w)
    return (lo+hi)/2
# full-corpus presence counts already in R['rq1_fullcorpus']; rebuild pres from full.csv
from collections import Counter as _C, defaultdict as _dd
_fam=lambda l:l.split(".")[0]
_ARM={"human":("human","zero_shot"),"clean":("gpt-3.5-turbo-0125","zero_shot"),"modern":("gpt-5.4-nano","zero_shot")}
_fams=[f"GE{i}" for i in range(1,7)]+[f"AE{i}" for i in range(1,7)]
_pres=_dd(_C)
for r in full:
    a=next((k for k,(m,s) in _ARM.items() if r["model"]==m and r["stage"]==s),None)
    if not a: continue
    for f in {_fam(x) for x in r["labels"].split(",") if x and x!="UNCOVERED"}: _pres[a][f]+=1
R["rq1_power"]={"full":{},"gold_mde":None,"full_mde":{}}
for pr in [("human","clean"),("human","modern"),("clean","modern")]:
    chi,df,N,V,p=_chipV(*pr,_pres,_fams)
    R["rq1_power"]["full"]["-".join(pr)]={"chi2":round(chi,1),"df":df,"N":N,"V":round(V,3),"p":p}
    R["rq1_power"]["full_mde"]["-".join(pr)]=round(_mde(df,N),3)
# gold MDE: ~125 bugs/arm, multi-label -> ~250 label-presences per pair, df ~ 8-11
R["rq1_power"]["gold_mde"]=round(_mde(11,250),3)
print("full p/V:",{k:(v["V"],f"{v['p']:.1e}") for k,v in R["rq1_power"]["full"].items()})
print("gold MDE (V@80%):",R["rq1_power"]["gold_mde"]," full MDE:",R["rq1_power"]["full_mde"])
Path("paper/results.json").write_text(json.dumps(R,indent=1)); print("saved")


full p/V: {'human-clean': (0.112, '3.2e-25'), 'human-modern': (0.082, '1.0e-11'), 'clean-modern': (0.093, '7.0e-05')}
gold MDE (V@80%): 0.259  full MDE: {'human-clean': 0.037, 'human-modern': 0.039, 'clean-modern': 0.064}
saved
